# Complement Validation - Train on WikiAtomicEdits (complement_wiki)

**Before running:** Settings -> Accelerator -> **T4 GPU**, Internet -> **On**.

Trains the FIXED ComplementGenerator (match-scaled subtraction) on WikiAtomicEdits,
where B = A + an inserted phrase (HIGH overlap). Purely self-supervised
(reconstruction of B from A+edge + InfoNCE diversity); the gold inserted phrase is
NEVER a training target.

**Data downloads from public GCS in-notebook** - no Google Drive needed for data.
Only the trained checkpoint optionally goes to Drive (rclone, RCLONE_CONF secret).

**Watch `collapse_sim`:** unlike MuSiQue (stuck ~0.89), here it SHOULD drop -- on
high-overlap data the true complement differs from encode(B).

In [ ]:
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/complement_wiki"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"
MAX_EXAMPLES    = 120000   # filtered multi-word-insertion pairs to use
UPLOAD_TO_DRIVE = True

In [ ]:
# 1. GPU check
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> T4 GPU")
if torch.cuda.get_device_properties(0).major < 7:
    raise RuntimeError("GPU is P100 - change to T4")
print(torch.cuda.get_device_name(0), "OK")

In [ ]:
# 2. Clone / pull repo + deps
import os
root = f"/kaggle/working/{REPO_NAME}"
os.system(f"git clone {REPO_URL} {root}" if not os.path.isdir(root) else f"cd {root} && git pull")
os.chdir(WORK_DIR)
print("cwd:", os.getcwd())
os.system("pip install -q 'transformers>=4.35.0' gdown")
print("deps ready")

In [ ]:
# 3. Download WikiAtomicEdits via robust Python downloader (wget can fail silently on Kaggle)
import sys, os
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
sys.path.insert(0, f"{WORK_DIR}/../retrieval_v3")
sys.path.insert(0, f"{WORK_DIR}/../retrieval_v2")
import data_wikiedits as dw
dw.download()   # urllib, deletes any 0-byte leftover, verifies size, raises on failure
print("data:", os.path.getsize(dw.GZ_PATH)/1e6, "MB")

In [ ]:
# 4. Smoke test (tiny, fast)
import os
os.chdir(WORK_DIR)
os.system("python train_wikiedits.py --smoke")

In [ ]:
# 5. Full training
import os
os.chdir(WORK_DIR)
log = "/kaggle/working/wiki_train.log"
os.system(f"python train_wikiedits.py --max_examples {MAX_EXAMPLES} 2>&1 | tee {log}")
print("\ntraining complete")

In [ ]:
# 6. Verify + optional rclone upload of checkpoint to Drive
import os, shutil
best = f"{WORK_DIR}/models/complement_wiki_best.pt"
if os.path.exists(best):
    print("checkpoint:", os.path.getsize(best)/1e6, "MB")
    shutil.copy(best, "/kaggle/working/complement_wiki_best.pt")
else:
    print("NOT FOUND - check log")

if UPLOAD_TO_DRIVE and os.path.exists(best):
    try:
        from kaggle_secrets import UserSecretsClient
        conf = UserSecretsClient().get_secret("RCLONE_CONF")
        os.makedirs("/root/.config/rclone", exist_ok=True)
        open("/root/.config/rclone/rclone.conf", "w").write(conf)
        os.system("curl -s https://downloads.rclone.org/rclone-current-linux-amd64.zip -o /tmp/r.zip")
        os.system("cd /tmp && unzip -qo r.zip && cp rclone-*-linux-amd64/rclone /usr/bin/ && chmod +x /usr/bin/rclone")
        rc = os.system(f"rclone copy {best} gdrive: --drive-root-folder-id {DRIVE_FOLDER_ID} -P")
        print("uploaded to Drive" if rc == 0 else f"rclone exit {rc}")
    except Exception as e:
        print("upload skipped:", e)

---
Next: run **eval_kaggle.ipynb** to measure whether the edge recovers the inserted phrase
(the decisive recovery test with the baseline ladder).